# Verl Quickstart (Foolproof Demo)

This notebook is designed to be **click-and-run** for first-time users.

## What you get
- A minimal **LLM inference demo** using `transformers` (downloads the model on first run)
- A minimal **Verl import + sanity checks**
- Clear steps and readable outputs

## Important
- **No model weights are bundled** in this image.
- The model will be **downloaded only when you run the inference cell**.
- If your network requires auth, you may need to `huggingface-cli login`.


## 0) Environment check
Run this first to confirm Python, GPU, and key environment variables.

In [ ]:
import os
import sys
import subprocess
import torch

print('python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('HF_HOME:', os.getenv('HF_HOME'))
print('TRANSFORMERS_CACHE:', os.getenv('TRANSFORMERS_CACHE'))
print('HF_HUB_CACHE:', os.getenv('HF_HUB_CACHE'))

try:
    out = subprocess.check_output(['nvidia-smi', '-L'], text=True)
    print('nvidia-smi -L:\n', out)
except Exception as e:
    print('nvidia-smi not available:', repr(e))

## 1) Verl sanity check
This verifies the package is installed and importable.

> If this fails, the image is missing Verl or your environment is not activated.

In [ ]:
import verl
print('verl imported OK')
print('verl version:', getattr(verl, '__version__', 'unknown'))

## 2) (Optional) Hugging Face login
Run this cell only if you need access to gated models or hit auth errors.

**You can skip this for public models.**

In [ ]:
import shutil
from IPython.display import Markdown, display

if shutil.which('huggingface-cli') is None:
    display(Markdown('`huggingface-cli` not found. If needed, install with: `pip install -U huggingface_hub`'))
else:
    display(Markdown('Run in terminal (recommended): `huggingface-cli login`'))
    display(Markdown('Or run the command below if you understand the security implications.'))
    print('huggingface-cli is available')

# Uncomment to run (not recommended inside notebook):
# !huggingface-cli login

## 3) One-click LLM inference demo (downloads model on first run)

This cell does the simplest possible thing:
- Downloads a small model
- Runs one generation
- Prints the output

### Default demo model
- `Qwen/Qwen2.5-0.5B-Instruct`

You can change `MODEL_ID` to anything compatible with `transformers`.


In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
PROMPT = 'Explain reinforcement learning in one paragraph.'

t0 = time.time()
print('loading:', MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype='auto',
    device_map='auto',
    trust_remote_code=True,
)

inputs = tokenizer(PROMPT, return_tensors='pt')
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.inference_mode():
    out = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
    )

text = tokenizer.decode(out[0], skip_special_tokens=True)
print('\n--- OUTPUT ---\n')
print(text)
print('\n--- STATS ---\n')
print('elapsed_sec:', round(time.time() - t0, 2))
print('device:', model.device)

## 4) What Verl is used for (practical next step)

Verl is typically used to run **post-training / RL-style optimization**.

A practical workflow looks like:
1. **Policy model** generates responses
2. A **reward function / reward model** scores them
3. A **trainer** updates the policy (PPO/GRPO/etc.)

This notebook intentionally stops before training to keep it safe and foolproof.

### Next step (recommended)
- Use the output above to confirm your environment works
- Then run a real Verl training job via CLI/scripts (recommended for real runs)


## 5) Troubleshooting (common)

- **OOM / CUDA out of memory**: pick a smaller model or reduce `max_new_tokens`.
- **Auth error**: run `huggingface-cli login` in terminal.
- **Slow download**: first run downloads model weights; subsequent runs are faster.
- **CPU only**: confirm your instance has GPU and CUDA is available.
